In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_path = "/content/drive/MyDrive/spam_classifier"
os.makedirs(save_path, exist_ok=True)
print("Drive ready. Everything will be saved here:", save_path)

Mounted at /content/drive
Drive ready. Everything will be saved here: /content/drive/MyDrive/spam_classifier


In [2]:
import pandas as pd
import numpy as np
import string
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])
print(df.shape)
df.head()

(5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

df['clean_message'] = df['message'].apply(clean_text)

In [5]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_message'])
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
param_grid = {'C': [0.1, 1, 10, 100]}
grid = GridSearchCV(SVC(kernel='linear', probability=True), param_grid, scoring='f1', cv=5)
grid.fit(X_train, y_train)

best_svm = grid.best_estimator_
print("Best C:", grid.best_params_)
print(classification_report(y_test, best_svm.predict(X_test), target_names=["ham", "spam"]))

Best C: {'C': 10}
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.98      0.86      0.92       149

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [7]:
%%writefile app.py
import streamlit as st
import joblib
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
import os

nltk_data_path = os.path.join(os.path.expanduser("~"), "nltk_data")
nltk.data.path.append(nltk_data_path)
nltk.download('stopwords', download_dir=nltk_data_path)
nltk.download('punkt', download_dir=nltk_data_path)
nltk.download('punkt_tab', download_dir=nltk_data_path)

model = joblib.load("spam_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

st.set_page_config(page_title="Spam Email Classifier", page_icon="📧")
st.title("📧 Spam Email Classifier")
st.write("Enter a message below and I'll tell you if it's Spam or Ham (not spam).")

user_input = st.text_area("Type or paste a message here:")

if st.button("Classify"):
    if user_input.strip() == "":
        st.warning("Please enter a message first.")
    else:
        cleaned = clean_text(user_input)
        vectorized = vectorizer.transform([cleaned])
        prediction = model.predict(vectorized)[0]
        probability = model.predict_proba(vectorized)[0]
        if prediction == 1:
            st.error(f"🚨 This looks like SPAM ({probability[1]*100:.1f}% confidence)")
        else:
            st.success(f"✅ This looks like HAM — not spam ({probability[0]*100:.1f}% confidence)")

Writing app.py


In [8]:
!cp app.py {save_path}/app.py
print("app.py saved permanently to Drive.")

app.py saved permanently to Drive.


In [9]:
with open(f"{save_path}/requirements.txt", "w") as f:
    f.write("streamlit\nscikit-learn\nnltk\njoblib\npandas\nnumpy\n")
print("requirements.txt saved.")

requirements.txt saved.
